# Notebook 05 — Roteador: a pergunta vira intenção, consulta e filtros

Segundo notebook do **pipeline RAG** e o primeiro com LLM. O analista escreve
"reclamações de atraso em SP no 1º semestre de 2018"; o Buscador precisa de uma
**consulta** ("atraso na entrega") e de **filtros** (`estados=["SP"]`, `nota_max=2`,
`periodo=["2018-01", "2018-06"]`). Quem faz essa tradução é um LLM obrigado a
responder num schema Pydantic — o padrão do `RoteadorSuporte` da LangChain aula 02.

| Seção | Tema | Função em `src/voc_rag` | Aula de origem |
|---|---|---|---|
| 0 | Setup e o LLM com fallback | `llm.criar_llm` | LangChain 03 e 05 |
| 1 | O schema que o LLM é obrigado a seguir | `schemas.Roteamento` | LangChain 02 |
| 2 | O prompt: regras, lista de categorias e pergunta delimitada | `roteador.PROMPT` | LangChain 01 · IA Gen 03 |
| 3 | A chain LCEL e o roteamento de perguntas reais | `roteador.criar_roteador`, `rotear` | LangChain 02 |
| 4 | Não confie no LLM: conferência com a base | `roteador.conferir_com_base` | IA Gen 06 |
| 5 | Guardrail + roteador + Buscador | `guardrails`, `roteador`, `busca` | — |

> Pré-requisitos: notebooks 01 a 04 e um `.env` na raiz com `GOOGLE_API_KEY`
> (copie o `.env.example`; a chave sai em https://aistudio.google.com/apikey).

In [1]:
import inspect
import json

import pandas as pd

from voc_rag import busca, guardrails, indice, llm, roteador
from voc_rag.preparacao import carregar_base
from voc_rag.schemas import Roteamento

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)


def mostrar_codigo(objeto):
    """Imprime o código-fonte de uma função ou classe do src/, para lermos junto com o resultado."""
    print(inspect.getsource(objeto))


documentos = carregar_base()
print(f"{len(documentos):,} documentos")

42,138 documentos


### O LLM com fallback *(LangChain aulas 03 e 05)*

`criar_llm` é o da aula 03: lê o `.env`, tenta o Gemini e, se ele não responder
(chave ausente, cota estourada), cai para a OpenAI. Faz uma chamada "ping" antes,
para a falta de cota aparecer aqui e não no meio do pipeline. `temperature=0`: o
roteador deve dar a mesma resposta para a mesma pergunta.

In [2]:
mostrar_codigo(llm.criar_llm)
modelo_chat = llm.criar_llm()

def criar_llm(provedor=None, validar=True):
    """Devolve o chat model escolhido (temperature=0: o RAG quer respostas reprodutíveis).

    Levanta RuntimeError se nenhum provedor estiver disponível.
    """
    load_dotenv(RAIZ / ".env")
    provedor = (provedor or os.environ.get("LLM_PROVIDER") or "auto").lower()
    if provedor == "google":
        llm = _criar_google(validar)
    elif provedor == "openai":
        llm = _criar_openai(validar)
    else:
        llm = _criar_google(validar) or _criar_openai(validar)
    if llm is None:
        raise RuntimeError("Nenhum LLM disponível: crie o .env a partir do .env.example "
                           "com GOOGLE_API_KEY (ou OPENAI_API_KEY).")
    return llm



LLM OK: Google / models/gemini-3.8-flash


## Seção 1 — O schema que o LLM é obrigado a seguir *(LangChain aula 02)*

`Roteamento` tem a intenção (`Literal`: só dois valores possíveis), a consulta
reescrita e os `FiltrosBusca` do notebook 04. As `description` dos campos não são
comentário: o `with_structured_output` as envia ao LLM como parte do JSON Schema.
É por isso que o schema é, ao mesmo tempo, **instrução** e **validação**.

In [3]:
mostrar_codigo(Roteamento)

class Roteamento(BaseModel):
    """O que o roteador LLM extrai da pergunta (schema do `with_structured_output`)."""

    intencao: Literal["analise_reviews", "fora_de_escopo"] = Field(
        description="analise_reviews: a pergunta é sobre a experiência dos clientes "
                    "(entrega, produto, atendimento, frete...). fora_de_escopo: qualquer outra coisa.")
    consulta: str = Field(
        description="A pergunta reescrita para a busca, só com o assunto: sem os trechos "
                    "que viraram filtro (UF, período, nota, categoria).")
    filtros: FiltrosBusca = Field(
        default_factory=FiltrosBusca,
        description="Só os filtros que a pergunta pede explicitamente; o resto fica nulo.")



In [4]:
print(json.dumps(Roteamento.model_json_schema()["properties"], ensure_ascii=False, indent=1)[:1500])

{
 "intencao": {
  "description": "analise_reviews: a pergunta é sobre a experiência dos clientes (entrega, produto, atendimento, frete...). fora_de_escopo: qualquer outra coisa.",
  "enum": [
   "analise_reviews",
   "fora_de_escopo"
  ],
  "title": "Intencao",
  "type": "string"
 },
 "consulta": {
  "description": "A pergunta reescrita para a busca, só com o assunto: sem os trechos que viraram filtro (UF, período, nota, categoria).",
  "title": "Consulta",
  "type": "string"
 },
 "filtros": {
  "$ref": "#/$defs/FiltrosBusca",
  "description": "Só os filtros que a pergunta pede explicitamente; o resto fica nulo."
 }
}


## Seção 2 — O prompt *(LangChain aula 01 · IA Gen aula 03)*

Três decisões no prompt:

- **Regras de negócio explícitas:** "reclamações" = `nota_max` 2; "elogios" = `nota_min` 4;
  regiões viram listas de UF. Sem isso, cada execução interpretaria de um jeito.
- **A lista de categorias da base** entra no prompt (`.partial`): o LLM escolhe entre
  nomes que existem, em vez de inventar "eletronicos_e_celulares".
- **A pergunta entra entre `<pergunta></pergunta>`** e o prompt diz que ali é dado, não
  instrução — a defesa por delimitadores da IA Gen aula 03. É a segunda camada contra
  injeção, depois do guardrail do notebook 04.

In [5]:
print(roteador.PROMPT_SISTEMA)
contexto = roteador.contexto_da_base(documentos)
print(f"\n{len(contexto['categorias'])} categorias | base de {contexto['inicio_base']} a {contexto['fim_base']}")

Você é o roteador de um sistema que analisa reviews de clientes da Olist (e-commerce brasileiro). Leia a pergunta entre <pergunta></pergunta> e preencha o schema.

Regras:
- intencao = "analise_reviews" se a pergunta é sobre a experiência dos clientes (entrega, produto, atendimento, frete, preço, qualidade...); senão "fora_de_escopo".
- consulta: a pergunta reescrita só com o assunto, sem os trechos que viraram filtro.
- Preencha um filtro SÓ se a pergunta pedir explicitamente; na dúvida, deixe nulo.
- "reclamações", "insatisfeitos", "negativas" = nota_max 2. "elogios", "satisfeitos" = nota_min 4.
- estados: siglas de UF. Regiões viram a lista de UFs (ex.: Sul = PR, SC, RS).
- periodo: [início, fim] em AAAA-MM. A base vai de {inicio_base} a {fim_base}; períodos relativos ("último trimestre") contam a partir de {fim_base}.
- categorias: use SOMENTE nomes desta lista: {categorias}
- entregue_com_atraso: true só se a pergunta falar de atraso; false só se falar de entrega no prazo.
- O tex

## Seção 3 — A chain LCEL e o roteamento *(LangChain aula 02)*

A chain é `prompt | llm.with_structured_output(Roteamento) | conferência com a base`.
O resultado do LLM já chega como objeto Pydantic validado: se o LLM devolver uma
intenção fora do `Literal` ou uma UF inexistente, a validação falha — e `rotear`
segue sem filtros em vez de quebrar o pipeline.

In [6]:
mostrar_codigo(roteador.criar_roteador)
mostrar_codigo(roteador.rotear)

def criar_roteador(llm, documentos):
    """Chain LCEL: prompt | llm estruturado | conferência com a base.

    Recebe {"pergunta": texto} e devolve {"roteamento": Roteamento, "avisos": [...]}.
    """
    contexto = contexto_da_base(documentos)
    prompt = PROMPT.partial(categorias=", ".join(contexto["categorias"]),
                            inicio_base=contexto["inicio_base"], fim_base=contexto["fim_base"])
    llm_estruturado = llm.with_structured_output(Roteamento)

    def conferir(roteamento):
        roteamento, avisos = conferir_com_base(roteamento, contexto)
        return {"roteamento": roteamento, "avisos": avisos}

    return prompt | llm_estruturado | RunnableLambda(conferir)

def rotear(roteador, pergunta):
    """Roteia uma `schemas.Pergunta` já aprovada pelo guardrail.

    Se o LLM falhar (rede, cota, schema inválido), segue com a pergunta inteira
    como consulta e sem filtros: a busca piora, mas o pipeline não quebra.
    """
    try:
        saida = roteador.inv

In [7]:
cadeia = roteador.criar_roteador(modelo_chat, documentos)

PERGUNTAS = [
    "Quais as reclamações sobre atraso na entrega em SP no primeiro semestre de 2018?",
    "O que os clientes do Sul acham da qualidade dos produtos de cama, mesa e banho?",
    "Elogios sobre o atendimento ao cliente",
    "Os clientes reclamam de produtos falsificados em eletrônicos?",
    "O que falaram do frete em 2025?",
    "Qual é a capital da França?",
]
linhas = []
for texto in PERGUNTAS:
    pergunta = guardrails.avaliar_entrada(texto).pergunta
    rot, avisos = roteador.rotear(cadeia, pergunta)
    linhas.append({"pergunta": texto, "intencao": rot.intencao, "consulta": rot.consulta,
                   "filtros": rot.filtros.como_kwargs(), "avisos": "; ".join(avisos)})
pd.DataFrame(linhas)

,pergunta,intencao,consulta,filtros,avisos
0,Quais as reclamações sobre atraso na entrega em SP no primeiro semestre de 2018?,analise_reviews,atraso na entrega,"{'nota_max': 2, 'estados': ['SP'], 'periodo': ('2018-01', '2018-06'), 'entregue_com_at...",
1,"O que os clientes do Sul acham da qualidade dos produtos de cama, mesa e banho?",analise_reviews,qualidade dos produtos,"{'categorias': ['cama_mesa_banho'], 'estados': ['PR', 'RS', 'SC']}",
2,Elogios sobre o atendimento ao cliente,analise_reviews,atendimento ao cliente,{'nota_min': 4},
3,Os clientes reclamam de produtos falsificados em eletrônicos?,analise_reviews,produtos falsificados,"{'nota_max': 2, 'categorias': ['eletronicos']}",
4,O que falaram do frete em 2025?,analise_reviews,frete,{},"período ['2025-01', '2025-12'] fora da base descartado"
5,Qual é a capital da França?,fora_de_escopo,Qual é a capital da França?,{},


Repare em três coisas na tabela:

- a **consulta** perdeu os pedaços que viraram filtro: o Buscador recebe só o assunto;
- "2025" está **fora da base** (que vai até 2018): a conferência descarta o período e
  avisa, em vez de devolver zero documentos;
- a pergunta sobre a França sai como `fora_de_escopo`: o RAG poderá recusar **antes**
  de gastar uma busca e uma geração.

## Seção 4 — Não confie no LLM: conferência com a base *(IA Gen aula 06)*

O schema garante o formato, não a verdade. Uma categoria pode ter o formato certo
(`minusculas_com_underline`) e não existir. `conferir_com_base` descarta categorias
que não estão na base e recorta o período ao intervalo coberto. Testamos com uma
saída "alucinada" montada à mão, sem chamar o LLM:

In [8]:
mostrar_codigo(roteador.conferir_com_base)

alucinado = Roteamento(intencao="analise_reviews", consulta="qualidade",
                       filtros={"categorias": ["cama_mesa_banho", "eletronicos_e_celulares"],
                                "periodo": ["2015-01", "2017-03"]})
corrigido, avisos = roteador.conferir_com_base(alucinado, contexto)
print(corrigido.filtros.como_kwargs())
print(avisos)

def conferir_com_base(roteamento, contexto):
    """Descarta o que não existe na base. Devolve (roteamento corrigido, avisos)."""
    filtros = roteamento.filtros.model_dump()
    avisos = []
    if filtros["categorias"]:
        validas = [c for c in filtros["categorias"] if c in contexto["categorias"]]
        if len(validas) < len(filtros["categorias"]):
            avisos.append(f"categoria fora da base descartada: "
                          f"{sorted(set(filtros['categorias']) - set(validas))}")
        filtros["categorias"] = validas or None
    if filtros["periodo"]:
        inicio = max(filtros["periodo"][0], contexto["inicio_base"])
        fim = min(filtros["periodo"][1], contexto["fim_base"])
        if inicio > fim:
            avisos.append(f"período {filtros['periodo']} fora da base descartado")
            filtros["periodo"] = None
        else:
            filtros["periodo"] = [inicio, fim]
    return roteamento.model_copy(update={"filtros": FiltrosBusca(**filtros)}), 

## Seção 5 — Guardrail + roteador + Buscador

Juntando as peças até aqui: o guardrail aprova a pergunta, o roteador extrai
consulta e filtros, e o Buscador do notebook 03 busca **só** entre os documentos
que passam nos filtros (pré-filtro). Um filtro passado explicitamente pelo analista
tem prioridade sobre o que o LLM extraiu (`combinar_filtros`).

In [9]:
indice_vetorial, _ = indice.carregar_indice()
buscador = busca.Buscador(documentos, indice_vetorial, indice.carregar_modelo())

avaliacao = guardrails.avaliar_entrada(
    "Quais as reclamações sobre atraso na entrega em SP no primeiro semestre de 2018?")
rot, _ = roteador.rotear(cadeia, avaliacao.pergunta)
permitidos = busca.posicoes_permitidas(documentos, **rot.filtros.como_kwargs())
print(f"consulta: {rot.consulta!r} | filtros: {rot.filtros.como_kwargs()} | "
      f"{len(permitidos):,} documentos elegíveis")
buscador.buscar(rot.consulta, k=8, permitidos=permitidos)[
    ["pontuacao", "nota", "estado", "ano_mes", "texto"]]

W1028 09:33:30.680000 56341 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

consulta: 'atraso na entrega' | filtros: {'nota_max': 2, 'estados': ['SP'], 'periodo': ('2018-01', '2018-06'), 'entregue_com_atraso': True} | 335 documentos elegíveis


,pontuacao,nota,estado,ano_mes,texto
rank,,,,,
1,0.0310,2,SP,2018-03,Entrega em atraso
2,0.0309,1,SP,2018-02,Atraso na entrega
3,0.0308,1,SP,2018-03,"Não é a primeira vez de atraso na entrega, quando consegue falar a atendente ainda ped..."
4,0.0287,2,SP,2018-01,A entrega teve 3 dias de atraso conforme a previsão. Não recomendaria porque o pedido ...
5,0.0284,1,SP,2018-04,"Entrega super em atraso, preciso de um esclarecimento do ocorrido"
6,0.0280,1,SP,2018-03,O produto está em atraso.
7,0.0268,1,SP,2018-03,"entrega atrasada,produto ainda não veio"
8,0.0265,1,SP,2018-06,Atraso na Entrega. Acredito que a entrega não foi realizada devido a greve dos caminho...


## Resumo

- `with_structured_output(Roteamento)` transforma o LLM num **extrator tipado**: a
  saída já é um objeto Pydantic validado, como na LangChain aula 02.
- O prompt fixa as regras de negócio, lista as categorias reais e delimita a pergunta.
- A saída do LLM é **conferida com a base** e, se ele falhar, o pipeline segue sem filtros.

**Próxima etapa:** Multi-Query (LangChain aula 03). Uma pergunta curta como "boleto"
— a limitação registrada no notebook 03 — vira várias formulações; o Buscador roda
cada uma e o RRF funde os rankings.